*Your Name*

*Collaborator's Names*

# Improving Performance with NumPy and Pandas

NumPy, Pandas, and other packages in the scientific Python ecosystem provide a huge variety of tools that make common research tasks simpler. Generally, its best to use these tools as they have been specially optimized; however, there can be reasons to prefer code that takes slightly longer to run. Sometimes performant code is hard to read. Other times, developers may have had other goals in mind, and don't prioritize performance. **Always, always test your code changes to ensure you're actually seeing an improvement** (and that your results are correct).

The code used in this workbook is modified from [High Performance Python by Gorelick & Ozsvald](https://github.com/mynameisfiber/high_performance_python_2e/tree/master/06_matrix).

---
## Refactoring Lists to NumPy Arrays



Last week we learned that NumPy arrays can take advantage of vectorization to efficiently perform mathematical operations on lost of (homogenous) data. The goal of this section is to provide experience converting an existing list-based code to use NumPy. It also introduces NumPy array slicing - another way NumPy leverages vectorization.

The code below simulates a simple 1D diffusion problem. Entries in a list function as cells in a grid,
where each cell's value represents the concentration of a fluid at a point in space. Over time, the concentration in each cell will change as the fluid redistributes. **You will be incrementally refactoring this list-based code to use NumPy.**

A small region near the center of the grid is given a higher concentration than its surroundings:

![Graph of the initial conditions for our sample 1D diffusion problem.](diffusion_ICs.png "Diffusion Initial Conditions")

After 500 iterations with the default parameters, the concentration smooths out:

![Graph of the diffusion problem after 500 iterations.](diffusion_updated.png "Diffusion After 500 Iterations")

### Algorithm Description

Each entry in the list `grid` corresponds to some coordinate along the $x$ axis, such that the $i$ th cell of `grid` corresponds to position $x_{i}$.
The concentration C at position $x_i$ --- or $C(x_i)$ ---  is then stored as `grid[i]`. 

Each time we update the grid --- taking a timestep `dt` --- we iterate over all the cells in the grid. To update the value of each cell,
we apply a "stencil" which combines information from neighboring cells in a weighted manner.

For cell $i$, the stencil uses data from cells $i-1$ and $i+1$ to approximate the second derivative of the concentration $C$:

$\frac{dC}{dt} = D \cdot \frac{d^2}{dx^2} C(x) \approx D \cdot \left( C(x_{i-1}) + C(x_{i+1}) - 2 C(x) \right)$

where $D$ is the diffusion coefficient.

Below is an example grid. The stencil is shown in blue. The red cells are special cells called "ghost zones." These cells exist so that the stencil can still be applied to the edges of the grid. Information in ghost zones is usually ignored when analyzing the results of simulations like this 1D diffusion problem, but are **very** important to the correct function of the diffusion algorithm.

![Diagram the stencil-based grid update and ghost zones](stencil.png)



In [1]:
import numpy as np

In [2]:
def run_simulation(grid_size = 600, 
                   dt = 0.1,
                   num_iterations = 500,
                   diffusion_coeff = 1.0):
    """
    Simulate the diffusion of a fluid in 1D using lists.
    
    grid_size: the number of cells in our grid.
    dt: time change for each iteration
    num_iterations: how many time steps we allow the fluid to diffuse.
    diffusion_coeff: diffusivity of the fluid; higher is more diffusive.

    Returns the state of the grid after num_iterations
    """

    # Construct 1D grid
    # Add an additional cell at each end to handle grid boundaries
    # grid[0] and grid[grid_size+1] are these boundary cells
    grid = [0.01] * (grid_size + 2)

    # Set the initial conditions
    # A small region (~10% of total length)
    # near the middle has high concentration
    start_index = int(grid_size * 0.4)
    end_index = int(grid_size * 0.5)
    for i in range(start_index+1, end_index+1):
        grid[i] = 0.02

    # Evolve the grid
    for t in range(num_iterations):

        # create a new grid to store updates
        new_grid = [0.0] * (grid_size + 2)

        # update main grid
        for i in range(1, grid_size+1):
            stencil = grid[i+1] + grid[i-1] - 2*grid[i]
            new_grid[i] = grid[i] + diffusion_coeff * stencil * dt

        # update boundary cells
        new_grid[0] = new_grid[1]
        new_grid[grid_size+1] = new_grid[grid_size]

        # swap new_grid and grid
        grid, new_grid = new_grid, grid

    return grid


---
### Exercises

1. Time `run_simulation` to establish a baseline. You can use the default arguments.

*Record timing here*

2. Use `lprun` to profile the existing list-based version of `run_simulation`. Even though all of our code is in a single function, you will still need to specify the `-f` argument. The following questions will require the profiling output to answer.

3. Which lines take the largest fraction of the overall runtime? What purpose do these lines serve?

*Answer here*

4. Which lines take the most amount of time *per hit*? Is there a common theme between these lines? Think about memory --- allocation, access, data movement, etc.

*Answer here*

5. Using a new function, convert `grid` and `new_grid` to NumPy arrays (recall the [array creation](https://numpy.org/doc/1.25/reference/routines.array-creation.html#numerical-ranges) routines). Check the results against the figures shown above and time your refactor. Is this version of the code slower or faster? Why do you think that is?

*Answer here*

6. Refactor the diffusion code a second time. This time, eliminate the `for` loops by using [array slicing](https://numpy.org/doc/stable/user/absolute_beginners.html#indexing-and-slicing). This will enable us to use vectorization.

Hint: the loop
```python
for i in range(1, grid_size+1):
    grid[i]
```
will become
```python
grid[1:-1]
```
Drawing a diagram may help you, especially when handling the ghost zones!

7. Time the slice-based code for 500 iterations. How does the speed of this change compare to both our previous versions? Is this what you expect?

*Answer here*

8. There is one last easy refactor we can do (if you are familiar with this style of algorithm, you may have already spotted it). Profile the vectorized 1D diffusion code and consider the lines with multiple hits. We have been able to reduce the time spent on some of these lines thanks to vectorization. Another strategy is simply to reduce the number of times a line is executed! Which line can benefit from this kind of optimization without breaking the algorithm?

*Answer here*

9. Refactor the code a final time to implement the optimization identified above. Compared to the *original* version, what is the final factor of speed-up achieved?

*Answer here*

---
## Operating on Pandas Rows

Pandas is a library that provides data structures and tools for manipulating complex datasets. The most commonly used data structure is the DataFrame. A common analogy for DataFrames is a spreadsheet.

The dataset `customer_hours.csv` contains a large amount (100,000 rows) of fake data representing customer cell phone call time over a two week period. The first column is the customer's unqiue ID number and each subsequent column is the number of hours the customer has spent on a call that day. We're interested in the general trend of each customer's cell usage: does it change over the two weeks or stay largely the same?

To do this, we can use ordinary least squares (OLS) to fit a line $y=mx+b$ to each customer's fake cell phone data. Row-by-row operations like this are commonly applied to Pandas DataFrames. Since we're only interested in how cell phone usage changes over time, we only need the slope $m$ from the OLS fit.

There are two parts to this task where we can explore performance: first, we need a function for performing OLS. Second, we need to apply this function to each row in the DataFrame.

### Ordinary Least Squares

OLS is an incredibly common algorithm, so much so that NumPy, SciPy, and scikit-learn all have their own implementations. How do they compare against each other? Using the following code as setup, you'll write and test a "wrapper" for NumPy's [`lstsq`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html), SciPy's [`linregress`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html), and scikit-learn's [`LinearRegression` class](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [3]:
import pandas as pd
from numpy.linalg import lstsq 
from scipy.stats import linregress
from sklearn.linear_model import LinearRegression

df = pd.read_csv("customer_data.csv", index_col="CustomerID")
test_row = df.iloc[0]

For reference, **the resulting slope should be around -0.0117.**

---
#### Exercises

1. Write wrapper functions for each of the routines named above. Each wrapper should take a row from a Pandas DataFrame as input and return the slope of the fitted line. Reference the documentation linked above as needed. 

Notes:
* For simplicity, you may use integers to represent the days (the independent variable) rather than the calendar date in the column label; e.g.,  `x = np.arange(row.shape[0])`. 
* Even if you don't fully understand what NumPy's `lstsq` is doing, the example matches well to our problem.
* scikit-learn is intended to be used for machine learning. The documentation references values `n_features` and `n_targets`; for our purposes, these are each 1. Additionally, the `y` value for the `fit` method is the unmodified data from `row`.

2. Time each function using `test_row`. Which method is fastest? Which is slowest?

*Answer here*

3. SciPy *also* offers a `lstsq` function in it's linear algebra module: `scipy.linalg.lstsq`. In fact, SciPy offers a number of ways to tackle our linear fitting problem. SciPy's `lstsq` behaves very similarly to NumPy's `lstsq`. Create a new wrapper for SciPy's `lstsq` based on your NumPy function.

In [4]:
from scipy.linalg import lstsq as sc_lstsq

4. Time the SciPy `lstsq` wrapper. How does it compare to the NumPy version? Using Google, can you find an explanation of the difference?

*Answer here*

5. In your opinion, which was the *easiest* OLS version to implement? How does the performance of this version compare to the others?

*Answer here*

6. In your opinion, which OLS version represents the best compromise between performance and ease of use? When thinking about "ease of use," consider not only your own time spent developing but *also* the readability of your code for others (including your future self).

**This question ended up being mostly redundant with question 5. They can probably be merged?**

*Answer here*

7. Let's find out why scikit-learn had the slowest OLS implementation. Profile `LinearRegression`'s fitting function using `lprun`. The `LinearRegression.fit` function is itself wrapped (that's what the `@` statement in the source code above the function definition indicates) so your call to `lprun` will need to look like `%lprun -f LinearRegression.fit.__wrapped__ <scikit-learn OLS wrapper>` (if you don't include the `__wrapped__` portion, `line_profiler` will likely suggest it near the top of your output). Which lines dominate the runtime by percentage?

*Answer here*

8.  Looking at the page for the [`LinearRegression` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression), you'll see a link called "source" near the top of the page. Clicking this link will take you to the line in the source code where `LinearRegression` is implemented! You can then click on other functions to see where they are implemented using the righthand sidebar. Use this to help you determine the purpose of the functions you identified above.

*Answer here*

9. Having done some investigating, why do you think scikit-learn has the slowest OLS algorithm? Even though it makes their code slow, why might the scikit-learn developers have chosen to write their function this way? What benefit does it provide users of scikit-learn?

*Answer here*

### Applying Functions to Rows

We've tested several functions for performing OLS on a row, now it's time to apply it to all rows in `customer_data.csv`.
We'll compare a "purely Pythonic" approach with the tools provided by Pandas.

For the following exercises, use the NumPy `lstsq` wrapper you wrote above.

---
#### Exercises

1. The most natively Pythonic approach is to use a `for` loop to iterate over the rows' numerical indices; i.e., to iterate through `range(df.shape[0])`. The rows can then be accessed by `df.iloc[i]` where `i` is the numerical index. Write a loop that applies your NumPy OLS wrapper to each row in the DataFrame we loaded earlier and collects each row's results in *either* a list or a NumPy array (you decide!). Then, time how long this loop takes to execute.

*Answer here*

2. Pandas offers the [`df.apply()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html) method which can apply a given function along either every column (`index=0`) or along every row (`index=1`). This function eliminates the need for a `for` loop, and automatically collects the results of our function. Implement `df.apply` and time how long it takes to execute.

*Answer here*

3. By default, `df.apply()` creates a separate Pandas Series object for each row. This is useful if we want to use Pandas' features in the function we are applying, like indexing by column name; yet, our OLS wrappers don't use any Pandas-specific features. We can access the raw underlying NumPy array with the `raw=True` argument. Add this argument to `df.apply()` and time the result.

*Answer here*

4. Does the `raw=True` argument in `df.apply()` lead to speed up for other kinds of wrappers? That is, do other OLS implementations benefit form working with the raw NumPy array? Pick one or more of your other wrappers and time them with and without `raw=True`.

*Answer here*

## Wrap-Up Reflection

The material we've covered in the last two weeks follows a particular theme: general-purpose, broadly-applicable data structures and functions versus programming tools built for comparatively narrowly-focused, specific goals. What are examples of both cases? What are the advantages of using more general tools, and what are the advantages of using specialized tools?

**Interestingly, a lot of people identified sklearn as a specialized tool (b/c of its machine-learning focus) and stated that specialized tools often provide better performance. These things are true separately, but they did not seem to synthesize that scikit learn was the slowest because of its additional data verification etc. In my mind this makes it a more general tool, as it does more for the user. Perhaps I need to reframe "special" vs "general". Students took away my desired point about specialized tools being more performant, but did not classify tools the same way I would (NumPy was also often cited as a general tool, which is true, except in comparison to lists)**

*Answer here*

## For Next Class

Next week we'll start working with multiple CPU cores. A common way to do this is to use *threads* - multiple streams of program execution that share memory. It's this shared memory aspect that makes threads an attractive way to run a program on multiple CPUs; however, Python's design means threads aren't as helpful as they are in other languages. This design feature is called the Global Interpreter Lock (GIL), and the following pre-class resources are intended to make you more familiar with this parallel programming roadblock.

* What is an interpreter anyway? Read this overview of [compiled vs interpreted languages](https://www.freecodecamp.org/news/compiled-versus-interpreted-languages/). The article points out that Python can be run in compiled mode (if you've ever seen a `.pyc` file, that's compiled Python code) but this is usually reserved for imported modules.
* Watch this video [introducing the GIL](https://www.youtube.com/watch?v=XVcRQ6T9RHo).
* Read this article explaining [why the GIL is necessary](https://realpython.com/python-gil/).
* **Optional:** a recent Python Enhancement Proposal (PEP) lays out a road map for removing the GIL! If you're interested, you can read more [here](https://www.infoworld.com/article/3704248/python-moves-to-remove-the-gil-and-boost-concurrency.html).